In [ ]:
from graph.src.triplet_extraction import *
import phonlp



In [ ]:
vncorenlp_client = init_vncorenlp(r"D:\uit_chatbot\graph\nlp_models\VnCoreNLP-1.2")
phoNLP_model = phonlp.load(r"D:\uit_chatbot\graph\nlp_models\phonlp")
synonym_dict = load_synonym_dict(r"D:\uit_chatbot\graph\listSameKey.txt")
stopwords = load_stopwords(r"D:\uit_chatbot\graph\stopwords.csv")

In [ ]:
os.makedirs(os.path.dirname(r"D:\uit_chatbot\retrieval\logger.txt"), exist_ok=True)
logger, console_handler, file_handler = setup_logger(
    name="triplet_extraction",
    level=logging.DEBUG,
    log_to_file=True,
    file_path=r"D:\uit_chatbot\retrieval\logger.txt"
)

# Disable console logging (optional)
logger.removeHandler(console_handler)

logger.info("Starting triplet extraction...")
logger.debug("Debug mode enabled")

In [ ]:
def extract_triplets(sentence) -> [] :

    triplets = triplet_extraction(
                    text=sentence,
                    vncorenlp_client=vncorenlp_client,
                    phoNLP_model=phoNLP_model,
                    stopwords=stopwords,
                    logger=logger,
                )

    triplets_list = [
        {"c1": c1, "r": r, "c2": c2}
        for (c1, r, c2) in triplets
        if c1 and r and c2
    ]
    return triplets_list

In [ ]:
from retrieval.src.retrieval.triplet_retriever import TripletRetriever
retriever = TripletRetriever()

In [ ]:
triplets = extract_triplets("Sinh viên được  chuyển đổi tín chỉ đối với những môn học nào")
r = retriever.search_triplets(triplets)
print(r)

In [ ]:
import sqlite3

In [ ]:
conn = sqlite3.connect(r"D:\uit_chatbot\graph\jupyter\uit_law.db")
conn.row_factory = sqlite3.Row

In [ ]:
def get_law(law_id):
    cur = conn.cursor()
    cur.execute("SELECT * FROM laws WHERE id = ?", (law_id,))
    row = cur.fetchone()
    return dict(row) if row else None



In [ ]:
def get_law_with_parents(law_id):
    cur = conn.cursor()
    cur.execute("SELECT * FROM laws WHERE id = ?", (law_id,))
    row = cur.fetchone()

    if not row:
        return None

    node = dict(row)

    parent_id = node.get("parent_id")
    if parent_id:
        parent_node = get_law_with_parents(parent_id)
        node["__parent"] = parent_node
    else:
        node["__parent"] = None

    return node

In [ ]:
def flatten_hierarchy(node):
    order = []
    current = node

    while current:
        order.append(current)

        # đảm bảo "__parent" là object hoặc None
        parent = current.get("__parent")
        if parent is None or isinstance(parent, dict):
            current = parent
        else:
            # nếu lỡ là string hoặc loại khác → dừng để tránh lỗi
            break

    return list(reversed(order))

def generate_law_text(id):
    node = get_law_with_parents(id)
    nodes = flatten_hierarchy(node)

    parts = []
    tit = ""
    for n in nodes:
        title = (n.get("title") or "").strip()
        content = (n.get("content") or "").strip()

        if title:
         tit += title + " "
        if content and n == node:
            parts.append(tit)
            parts.append(content)

    return "\n".join(parts).strip()


In [ ]:
import pandas as pd



# 1. Đọc file gốc
df = pd.read_csv(r"D:\uit_chatbot\retrieval\testing.csv")

# 2. Thêm cột content
df["content"] = df["id"].apply(generate_law_text)
print(df)
# 3. Ghi đè file
df.to_csv(r"D:\uit_chatbot\retrieval\testing.csv", index=False)

In [ ]:
print(generate_law_text("9fc4ca12fa6a4f3deec9aa112eb4f8892b02b3a0c04af7845a4e791a446766f0"))

In [ ]:
import os

from groq import Groq
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("Chưa thiết lập GROQ_API_KEY trong file .env")
client = Groq(
    api_key = api_key,
)

def chat_groq(message):
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": message,
            }
        ],
        model="llama-3.3-70b-versatile",
    )
    return chat_completion.choices[0].message.content

In [ ]:

correct_prompt_template = """Ngữ cảnh của câu sau đây là câu hỏi của sinh viên về chương trình học. Hãy kiểm tra và sửa lỗi chính tả. Chỉ trả về câu đã sửa. Câu gốc: {text_correct_input}"""
def correct_text(text):
    prompt = correct_prompt_template.format(text_correct_input=text)
    corrected_text = chat_groq(prompt)
    return corrected_text

In [ ]:
summarize_text_prompt = """VAI TRÒ & MỤC TIÊU: Bạn là một AI chuyên gia trích xuất thông tin. Nhiệm vụ của bạn là nhận một văn bản thô từ người dùng và "làm phẳng" (flatten) nó thành các sự kiện cốt lõi để chuẩn bị cho tác vụ Tách Triplet (Triplet Extraction).

MỤC TIÊU CHÍNH: Đầu ra KHÔNG cần phải có ngữ nghĩa mượt mà cho người đọc. Mục tiêu là tạo ra một chuỗi dữ liệu mà máy tính có thể dễ dàng phân tích thành các bộ (Chủ thể - Quan hệ - Tân ngữ).

ĐẦU VÀO (INPUT): {text_input}

QUY TẮC BẮT BUỘC:

1.  **Loại bỏ "Nhiễu" (Noise Reduction):**
    * Loại bỏ tuyệt đối các từ đệm, từ thừa không mang nghĩa (ví dụ: thì, là, mà, à, ừm).
    * Loại bỏ các cụm từ mang tính hội thoại, cảm xúc, hoặc xin phép (ví dụ: "tôi lo lắng", "xin vui lòng", "bạn có thể cho tôi biết", "giúp tôi với").

2.  **Giữ lại "Tín Hiệu" (Signal Retention):**
    * Tập trung giữ lại các **thực thể chính** (ai, cái gì, ở đâu).
    * Tập trung giữ lại các **hành động/quan hệ** (làm gì, như thế nào).
    * Giữ lại các **điều kiện** quan trọng (thời gian, địa điểm, điều kiện nếu-thì).

3.  **Đảm bảo Chất lượng Đầu ra (Output Constraints):**
    * **Ngắn gọn:** Đầu ra phải ngắn hơn đáng kể so với đầu vào.
    * **Giảm độ phức tạp:** Chuyển các cấu trúc câu phức, rườm rà thành các cụm từ khóa hoặc câu đơn súc tích nhất có thể.
    * **Giảm nhiễu ngữ nghĩa:** Loại bỏ mọi chi tiết phụ không liên quan trực tiếp đến ý chính của câu hỏi.


4.  **PHÂN BIỆT BỐI CẢNH VÀ NỘI DUNG HỎI (QUAN TRỌNG NHẤT):**
    * Bạn **CHỈ ĐƯỢC** tóm tắt các sự kiện, bối cảnh, tình huống được cung cấp trong câu.
    * Bạn **PHẢI LOẠI BỎ** hoàn toàn nội dung, chủ đề, hoặc hành động đang được hỏi. **Tuyệt đối không** được biến phần câu hỏi thành một câu trần thuật/khẳng định.
---
QUY TRINH THỰC HIỆN: Hãy phân tích văn bản đầu vào và áp dụng các quy tắc sau một cách nghiêm ngặt:

PHẦN 1: QUY TẮC LỌC NỘI DUNG (GIỮ LẠI vs. LOẠI BỎ)

Bạn PHẢI LOẠI BỎ tất cả những điều sau:

Cảm xúc & Tính chủ quan: Bất kỳ từ ngữ nào thể hiện tâm trạng (tôi rất lo lắng, bức xúc, quá căng thẳng, tôi nghĩ là).

Đại từ nhân xưng: Tất cả các đại từ (tôi, chúng tôi, anh ấy, công ty họ). Hãy thay thế bằng vai trò pháp lý của họ (người lao động, người mua, bên A).

Thông tin cá nhân không liên quan: Tên riêng cụ thể, địa chỉ nhà, số điện thoại.

Từ ngữ đệm & Lặp lại: Các từ thừa, không mang ý nghĩa pháp lý (thì, là, mà, vấn đề là, chuyện là).

Bạn PHẢI GIỮ LẠI VÀ CHUẨN HÓA tất cả những điều sau:

Thời gian & Điều kiện (Các Thuộc tính):

Thời hạn/Mốc thời gian: 30 ngày, sau 2 năm, kể từ ngày 1/1/2024.

Điều kiện & Ngoại lệ: nếu không thông báo trước, trừ trường hợp bất khả kháng, khi tài sản bị hư hỏng.

PHẦN 2: QUY TẮC TÁI CẤU TRÚC CÂU

Mục tiêu là đơn giản hóa ngữ pháp để máy có thể dễ dàng phân tích quan hệ.

Chuyển đổi câu hỏi: KHÔNG tóm tắt câu hỏi. Thay vào đó, hãy tóm tắt sự kiện dẫn đến câu hỏi đó.

Đơn giản hóa câu: Chuyển đổi cấu trúc bị động thành chủ động khi có thể (ví dụ: "Người lao động bị công ty sa thải" -> "Công ty sa thải người lao động").

PHẦN 3: QUY TẮC ĐỊNH DẠNG ĐẦU RA (NGHIÊM NGẶT)

Đây là quy tắc bắt buộc để đảm bảo tính độc lập của dữ liệu cho ERE.

Chỉ sử dụng câu đơn: Toàn bộ đầu ra phải được chia thành các câu đơn. Mỗi câu chỉ mô tả một sự kiện, một mối quan hệ, hoặc một thuộc tính. Tuyệt đối không dùng câu ghép, câu phức.

Độc lập về ngữ nghĩa: Mỗi câu đơn phải hoàn toàn độc lập về mặt ý nghĩa. Người đọc (hoặc máy) phải hiểu được câu đó mà không cần đọc câu trước hoặc câu sau.

Không tham chiếu chéo: Tránh sử dụng đại từ (họ, nó, anh ta) hoặc các cụm từ tham chiếu (việc này, sau đó) để liên kết với các câu trước. Nếu cần, hãy lặp lại chủ thể một cách rõ ràng.


Ví dụ (SAI): "Bên A ký hợp đồng. Họ chưa thanh toán."

Ví dụ (ĐÚNG): "Bên A ký hợp đồng. Bên A chưa thanh toán."

Phân tách bằng dấu chấm: Mỗi câu đơn phải kết thúc bằng một dấu chấm (.). Các câu được đặt liền nhau, chỉ ngăn cách bởi dấu chấm và một khoảng trắng.


---
BẢN TÓM TẮT BỐI CẢNH (OUTPUT):
(Chỉ cung cấp văn bản tóm tắt đã được làm sạch theo các quy tắc 1, 2, và 3)"""
def summarize_text(text_input):
    prompt = summarize_text_prompt.format(text_input=text_input)
    summarized_text = chat_groq(prompt)
    return summarized_text

In [43]:
def handle_question(question):
    t1 = correct_text(question)
    t2 = summarize_text(t1)
    triplets  = extract_triplets(t2)
    r = retriever.search_triplets(triplets)
    return r[:5]

In [44]:
handle_question("Trong trường hợp em sắp tốt nghiệp và số tín chỉ cần học còn lại ít hơn 14 tín chỉ thì em có bắt buộc phải đăng ký đủ mức tối thiểu của một học kỳ chính không")

100%|██████████| 1/1 [00:00<00:00, 10.98it/s]


[{'doc_id': 'b73a9fce7622c7a1a4677ed8872d725734eb40b0ac7d7e1310e4b96e7a1cd164',
  'total_score': 10},
 {'doc_id': '78c086a2176617cb5e1a6d841cfab5fbef0a15d797d661f496783947a1ebc00a',
  'total_score': 6},
 {'doc_id': 'c8d11bf589bbfb1f37b4eaccfa5507966149f4fffcfc6d52483fcfc25f9c838f',
  'total_score': 4},
 {'doc_id': '343dad44854363f867a471e7bb8ffc7ce136b621c8195e492c97b683e62212fb',
  'total_score': 4},
 {'doc_id': '40e244aa82d9416c9a8905d0e14bb38769f4845ec5d412fbe208391545497929',
  'total_score': 4}]

In [ ]:
triplets = extract_triplets("Sinh viên sắp tốt nghiệp. Số tín chỉ cần học còn lại ít hơn 14 tín chỉ.")
print(triplets)
r = retriever.search_triplets(triplets)
id = r[0]["doc_id"]
generate_law_text(id)

In [ ]:
df = pd.read_csv(r"D:\uit_chatbot\retrieval\testing.csv")

# 2. Thêm cột content
df["content"] = df["id"].apply(generate_law_text)
print(df)
# 3. Ghi đè file
df.to_csv(r"D:\uit_chatbot\retrieval\testing.csv", index=False)

In [46]:
# Đọc file
import time

df = pd.read_csv(r"D:\uit_chatbot\retrieval\testing.csv")
for i in range(1, 6):
    df[f'top{i}'] = ""

failed_rows = []
for idx, row in df.iterrows():
    content = row["question"]

    if pd.isna(content) or content.strip() == "":
        print(f"Bỏ qua hàng {idx}: content rỗng")
        continue

    try:
        results = handle_question(content)
        for i in range(0,len(results)):
            doc_id = results[i]["doc_id"]
            text = "score: " + str(results[i]["total_score"]) + generate_law_text(doc_id)
            df.at[idx, f'top{i+1}'] = text

        print(f"Thành công hàng {idx}: {len(results)} cặp")

    except Exception as e:
        error_msg = f"Lỗi hàng {idx} (content: '{content[:50]}...'): {str(e)}"
        print(error_msg)
        failed_rows.append(idx)

    time.sleep(30)

print(f"Hàng thất bại: {failed_rows}")
print(df[['content', 'top1', 'top2', 'top3', 'top4', 'top5']].head())  # Kiểm tra

# Lưu file mới
df.to_csv(r"D:\uit_chatbot\retrieval\testing_with_tops.csv", index=False)
print("Đã lưu file!")


100%|██████████| 1/1 [00:00<00:00, 11.90it/s]


Thành công hàng 0: 5 cặp


100%|██████████| 1/1 [00:00<00:00, 11.89it/s]


Thành công hàng 1: 5 cặp


100%|██████████| 1/1 [00:00<00:00, 11.28it/s]


...

Thành công hàng 17: 5 cặp


100%|██████████| 1/1 [00:00<00:00, 13.42it/s]


Thành công hàng 18: 5 cặp


100%|██████████| 1/1 [00:00<00:00, 12.61it/s]


Thành công hàng 19: 5 cặp
Hàng thất bại: []
                                             content  \
0  chương 1 điều 6 khoản 1 điểm a \r\nĐào tạo trì...   
1  chương 1 điều 6 khoản 1 điểm b \r\nThời gian đ...   
2  chương 1 điều 7 khoản 1 điểm a \r\nĐáp ứng đượ...   
3  chương 1 điều 7 khoản 1 điểm b \r\nThể hiện rõ...   
4  chương 1 điều 7 khoản 1 điểm c \r\nĐược thiết ...   

                                                top1  \
0  score: 28chương 1 điều 6 khoản 1 điểm b \nThời...   
1  score: 17chương 1 điều 6 khoản 1 điểm b \nThời...   
2  score: 63chương 4 điều 30 khoản 1 \nTrường xây...   
3  score: 4chương 1 điều 7 khoản 2 điểm b \nKhối ...   
4  score: 3chương 1 điều 7 khoản 2 điểm b \nKhối ...   

                                                top2  \
0  score: 18chương 1 điều 6 khoản 2 \nTùy theo kh...   
1  score: 16chương 1 điều 3 khoản 4 \nKhoá luận t...   
2  score: 62chương 2 điều 18 khoản 2 điểm c \nĐã ...   
3  score: 2chương 5 điều 31 khoản 3 \nLàm khóa lu...   
4 